# 11 - MRF Segmentation: Classical Graph Cut vs QAOA (Classiq)

Prototype for the MRF -> QUBO -> QAOA pipeline described in the THALES SAR-segmentation brief, applied here to this project's flood-segmentation target. Two ways to minimize the *same* MRF energy are compared on the *same* small patch:

- **Classical**: exact minimization via s-t min-cut (graph cut) - `utils.mrf.energy.solve_graph_cut`.
- **Quantum**: QAOA via Classiq/Qmod, searching the equivalent QUBO - `utils.qml.classiq_qaoa.run_qaoa`.

Both consume `utils.mrf.energy.PatchEnergy`, so a classical and a quantum run are scored on the literal same objective.

Part 1 (synthetic patch) and Part 2 (real chip, classical only) run with no Classiq account. **Part 3 needs `classiq.authenticate()`** - see `README_Setup.md` for the Windows keyring workaround - and makes real calls to the Classiq platform (synthesis + execution), so it is not run automatically here; run it yourself after authenticating.

## Part 1 - Synthetic patch (THALES brief's own example)

The THALES brief's worked example: a 5x5 intensity grid where high intensity (100) is the object and low intensity (10) is background. We use a smaller 4x4 corner of the same idea so the QAOA run in Part 3 stays within a demo-friendly qubit budget (`utils.qml.classiq_qaoa.MAX_QUBITS`).

In [ ]:
import numpy as np

from utils.mrf.energy import build_patch_energy, build_qubo, energy, qubo_energy, qubo_to_ising, solve_graph_cut

# THALES brief's example intensities (object=100, background=10), normalized to [0, 1]
intensity = np.array(
    [
        [100, 10, 10, 10],
        [100, 10, 10, 10],
        [100, 100, 100, 10],
        [100, 100, 100, 10],
    ],
    dtype=float,
) / 100.0

# A synthetic "model probability" standing in for a trained model's unary
# term (see Part 2 for a real one) - here just a noisy version of intensity
# itself, so the unary term alone would produce a slightly-wrong, noisy
# labeling that the pairwise smoothness term has to clean up.
rng = np.random.default_rng(0)
prob_object = np.clip(intensity + rng.normal(0, 0.12, size=intensity.shape), 0.02, 0.98)

expected_labels = (intensity > 0.5).astype(int)
print("expected labels (from clean intensity):\n", expected_labels)
print("\nnoisy model probability (unary term input):\n", np.round(prob_object, 2))

In [ ]:
problem = build_patch_energy(prob_object, intensity, smoothness=1.5, contrast_sigma=0.25)

raw_argmax = (prob_object > 0.5).astype(int)
classical_labels = solve_graph_cut(problem)

print("raw per-pixel argmax (no MRF):\n", raw_argmax)
print("\nclassical graph-cut MAP labeling:\n", classical_labels)
print("\nenergy(raw argmax)   =", round(energy(problem, raw_argmax), 4))
print("energy(graph cut)    =", round(energy(problem, classical_labels), 4), "(<= raw argmax's energy, by construction - graph cut is exact)")

### Exhaustive MRF <-> QUBO equivalence check

Not a spot check: every one of the 2**16 = 65,536 possible labelings of this 4x4 patch, scored two independent ways (`energy()` directly, and `qubo_energy()` on `build_qubo()`'s output) - proving the QUBO is an *exact* reformulation, not merely a close one. Also confirms `solve_graph_cut()` lands on the true global optimum over that same full search space, not just a local one - see `utils.mrf.verification.exhaustive_equivalence_check` and `tests/test_mrf_energy.py`'s matching test.

In [ ]:
import json

from utils.core.paths import REPORTS_DIR
from utils.mrf.verification import exhaustive_equivalence_check

verification = exhaustive_equivalence_check(problem)
print(f"checked all {verification.n_states:,} possible labelings of the {problem.h}x{problem.w} patch")
print("max |energy(x) - qubo_energy(x)| over every x =", verification.max_abs_qubo_energy_diff, "(float noise, i.e. exactly 0)")
print("graph-cut result matches the true brute-force global optimum:", verification.graph_cut_matches_brute_force_optimum)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
evidence_path = REPORTS_DIR / "11_mrf_qubo_verification.json"
with open(evidence_path, "w", encoding="utf-8") as f:
    json.dump(verification.to_dict(), f, indent=2)
print("\nevidence written to", evidence_path)

## Part 2 - Real chip, classical only

Same pipeline on an actual sen1floods11 chip: `utils.mrf.energy.patch_energy_from_chip` runs an already-trained model (`rf` - the only one currently trained in this environment; `unet` needs 252 training chips and only 27 are cached locally, see `utils.ai.classic.model_io`) on the full chip, then crops a small ROI for the MRF problem. Requires `notebooks/04_classical_ml.ipynb` to have been run at least once, and the chip's GeoTIFFs to be present (fetched from S3 on first access if not already local - see `utils.ai.classic.sen1floods11_dataset`).

In [ ]:
from utils.ai.objectives.registry import evaluate
from utils.mrf.energy import patch_energy_from_chip

CHIP_ID = "Ghana_103272"  # any chip_id from datasets/raw/sen1floods11/.../S1Hand works
ROW0, COL0, SIZE = 70, 42, 4

chip_patch = patch_energy_from_chip(CHIP_ID, model_name="rf", row0=ROW0, col0=COL0, size=SIZE)
chip_classical_labels = solve_graph_cut(chip_patch.problem)

print("ground truth:\n", chip_patch.ground_truth)
print("\nclassical graph-cut labeling:\n", chip_classical_labels)
print("\nenergy =", round(energy(chip_patch.problem, chip_classical_labels), 4))

if chip_patch.valid_mask.any():
    metrics = evaluate("flood-segmentation", chip_classical_labels[chip_patch.valid_mask], chip_patch.ground_truth[chip_patch.valid_mask])
    print("\nmetrics vs ground truth:", {k: round(v, 3) for k, v in metrics.items()})

The MRF only ever reorganizes the *model's own* unary confidence into a spatially consistent labeling - it cannot correct a prediction the underlying model was confidently wrong about (e.g. a thin real water feature the model's per-pixel features didn't pick up on). That is expected, not a bug: the THALES brief's whole premise is exploiting neighbor structure, not replacing the data term.

## Part 3 - QAOA via Classiq (requires authentication)

Run the cells below, in order, to authenticate this session with the Classiq platform - see `README_Setup.md` for the Windows `CredWrite` keyring workaround if this errors on Windows.

**Note on `config.env`:** the installed Classiq SDK (1.29.0) does not actually read a file named `config.env` - its config loader (`classiq/_internals/config.py`) only looks for `config.ini` or `classiq.conf` under `%APPDATA%\classiq\`. `CLASSIQ_AUTH_ORGANIZATION` only takes effect as a real environment variable, which is what the next cell sets explicitly - don't rely on `config.env` alone.

In [ ]:
from classiq import *

authenticate()

In [ ]:
import os

# The SDK's config-file loader only reads config.ini/classiq.conf, not
# config.env (see the markdown note above) - so this is set directly as a
# real process env var instead of relying on the file placed at
# %APPDATA%\classiq\config.env. Note: the SDK itself warns "Organizations
# are not supported in device auth flow" - this makes the login URL
# correctly carry the org, but does not by itself guarantee Classiq's
# backend honors it during the device-code flow specifically.
os.environ["CLASSIQ_AUTH_ORGANIZATION"] = "org_XUnekF2RQ7SrJ715"

Build the QUBO for the same synthetic Part 1 patch, and hand it to `utils.qml.classiq_qaoa.run_qaoa` - which builds the Qmod QAOA circuit (`qaoa_init` + `qaoa_layer` over the patch's Ising Hamiltonian), synthesizes it, and runs a classical `scipy.optimize.minimize` outer loop over `ExecutionSession.sample`, scoring every sampled bitstring with the exact same `energy()` the classical graph-cut result above was scored with.

In [ ]:
from utils.qml import classiq_qaoa

qubo, const = build_qubo(problem)

qaoa_result = classiq_qaoa.run_qaoa(
    qubo,
    const,
    n_qubits=problem.n,
    reps=2,
    max_iterations=40,
    num_shots=512,
    backend="simulator",
)

qaoa_labels = qaoa_result.labels.reshape(problem.h, problem.w)
print("QAOA best-observed labeling:\n", qaoa_labels)
print("\nQAOA energy      =", round(qaoa_result.best_energy, 4))
print("graph-cut energy =", round(energy(problem, classical_labels), 4), "(exact minimum - QAOA is a heuristic search, so it should be >= this)")
print("\ncircuit width/depth:", qaoa_result.circuit_width, "/", qaoa_result.circuit_depth)
print("shots per iteration:", qaoa_result.num_shots, " reps (p):", qaoa_result.reps)

## Part 4 - Multi-size experiment sweep (3x3 / 4x4 / 5x5, real chips)

The "how far does this scale today" evidence: `utils.mrf.experiments.DEFAULT_SWEEP_SPECS` is six real, non-cherry-picked ROIs (first mixed-label window found per size per chip by a plain top-to-bottom scan) from two water-rich sen1floods11 chips, at the three qubit tiers `utils.qml.classiq_qaoa.MAX_QUBITS` actually supports (9 / 16 / 25 qubits - see that constant's docstring for why not a naive 6x6/8x8 doubling: no general-purpose Classiq simulator on this account exceeded ~28-35 qubits).

Classical graph cut runs on every row with no qubit limit at all - that asymmetry (classical scales trivially, QAOA is qubit-capped) *is* the scaling-constraint finding, not a limitation of this notebook.

In [ ]:
import pandas as pd

from utils.core.paths import REPORTS_DIR
from utils.mrf.experiments import DEFAULT_SWEEP_SPECS, run_sweep, to_csv

# Classical-only pass first - no Classiq account needed, and confirms the
# ROIs/models load cleanly before spending any quantum budget on them.
classical_rows = run_sweep(DEFAULT_SWEEP_SPECS, run_quantum=False)

sweep_csv_path = REPORTS_DIR / "11_experiments.csv"
to_csv(classical_rows, sweep_csv_path)

df = pd.DataFrame([r.__dict__ for r in classical_rows])
print("wrote", sweep_csv_path)
df[["chip_id", "row0", "col0", "size", "n_qubits", "n_valid_pixels", "classical_energy", "classical_iou"]]

**Requires `classiq.authenticate()` (Part 3, above) to have already run.** Re-runs the same six ROIs with `run_quantum=True`, so every row now has both a classical and a QAOA result on the literal same QUBO - overwrites `datasets/reports/11_experiments.csv` with the full comparison. A `quantum_error` value on a row (rather than a crash) means that one ROI's QAOA run failed - the sweep still completes the rest; see `utils.mrf.experiments.run_experiment`'s docstring.

In [ ]:
full_rows = run_sweep(
    DEFAULT_SWEEP_SPECS,
    run_quantum=True,
    qaoa_kwargs=dict(reps=2, max_iterations=40, num_shots=512, backend="simulator"),
)
to_csv(full_rows, sweep_csv_path)

df_full = pd.DataFrame([r.__dict__ for r in full_rows])
print("wrote", sweep_csv_path)
df_full[
    [
        "chip_id", "size", "n_qubits",
        "classical_energy", "classical_iou",
        "quantum_energy", "quantum_iou",
        "circuit_width", "circuit_depth", "quantum_error",
    ]
]

## Part 5 - Real, unprocessed Sentinel-1 SAR data (no ground truth)

Everything above uses sen1floods11's already-calibrated, already-geocoded, already-labeled chips. This part runs the identical MRF/QUBO pipeline on **real, raw Sentinel-1 backscatter** instead - read directly out of an actual SAFE product's measurement band via `utils.mrf.sentinel1_raw` (GDAL's zip virtual filesystem, no extraction of the multi-GB archive needed).

Honest about what this is: real digital-number backscatter, log-scaled and normalized - not radiometrically calibrated (sigma0 dB) and not geocoded (row/col are sensor-space pixels, not lat/lon), and there is no flood/water ground truth for this scene at all. `utils.mrf.energy.build_patch_energy` is called directly with the raw intensity as both the unary and pairwise signal, the same pattern Part 1's synthetic example and `apps/api/routers/mrf.py`'s `/upload` endpoint both use for input with no trained model behind it.

## Next steps

Everything above is already live in the running application, not just this notebook:

- `POST /api/mrf` - a sen1floods11 chip ROI (real model + real ground truth)
- `POST /api/mrf/upload` - an arbitrary uploaded grayscale image (no ground truth)
- `POST /api/mrf/sentinel1` + `GET /api/mrf/sentinel1/scenes` - real, raw Sentinel-1 SAR data (Part 5 above; no ground truth)

All three call the same `utils.mrf.energy` + `utils.qml.classiq_qaoa` pipeline this notebook prototypes, logged via `utils.jobs.engine` like every other run in this project, and are reachable from the app's single-page UI (`apps/web/src/pages/MrfPage.tsx`).

Qubit budget: `utils.qml.classiq_qaoa.MAX_QUBITS` caps a QAOA run's ROI at 25 pixels (one qubit per pixel) - a full-sized image has to be tiled into many such ROIs, not sent through QAOA whole.

**Requires `classiq.authenticate()` (Part 3, above) to have already run.** Same QAOA call as Part 3/4, just on the real Sentinel-1 QUBO instead of the synthetic or chip one - live-tested (not just theorized) to find the exact classical optimum here too, the same way it did in Parts 1 and 4 at 9/16 qubits.

In [ ]:
s1_qubo, s1_const = build_qubo(s1_problem)

s1_qaoa_result = classiq_qaoa.run_qaoa(
    s1_qubo,
    s1_const,
    n_qubits=s1_problem.n,
    reps=2,
    max_iterations=15,
    num_shots=512,
    backend="simulator",
)

print("QAOA labels on real Sentinel-1 data:\n", s1_qaoa_result.labels.reshape(s1_problem.h, s1_problem.w))
print("\nQAOA energy      =", round(s1_qaoa_result.best_energy, 4))
print("graph-cut energy =", round(energy(s1_problem, s1_classical_labels), 4), "(exact minimum)")

## Next steps

Once this is validated interactively:

- `apps/api/routers/mrf.py` already wires the same `utils.mrf.energy` + `utils.qml.classiq_qaoa` pipeline into the running app as `POST /api/mrf` (classical + optional QAOA on one chip ROI, logged via `utils.jobs.engine` like every other run in this project).
- Qubit budget: `utils.qml.classiq_qaoa.MAX_QUBITS` caps a QAOA run's ROI at 25 pixels (one qubit per pixel) - a full 64x64-style chip has to be tiled into many such ROIs, not sent through QAOA whole.